# Run Age at Onset (AAO) Regression for PD Carriers in AOU

- Last Updated: 7/25/2026
- Description: This notebook runs a regression with AAO as an outcome variable with ITSN1 coding variants extracted for this project.
- Data: All of Us Release 8

## Imports

In [1]:
import pandas as pd

## Run AAO regression with PLINK2 GLM

In [ ]:
## Try a pfile run instead
## Make sure to have AAO column in your covariate file
## Keep cases only. For that use a PHENO column in covariate file
import os
import subprocess


for ANCESTRY in ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "FIN", "SAS"]:
    try:
            PREFIX = f"{ANCESTRY}"
            GENE = 'ITSN1'
            CHROM = "chr21"
            WORK_DIR =  f'/home/jupyter/121825_ITSN1_Results/{ANCESTRY}_R8_wgs'
            DATA_DIR = f'/home/jupyter/data'
            WORKSPACE_DIR = '/home/jupyter/'
            TOOLS_DIR='/home/jupyter/tools'



            OUTPUT_DIR = f'/home/jupyter/121825_ITSN1_Results/PLINK_GLM_files'
            inputPfile = f"{WORK_DIR}/{CHROM}_{ANCESTRY}"
            outputGLM = f"{OUTPUT_DIR}/{PREFIX}.cases_only"
            samplesToKeep = f"{WORK_DIR}/{ANCESTRY}.samplestoKeep.txt"
            covariate = f"{WORK_DIR}/{ANCESTRY}_covariate_file.txt"
            extract = f"{WORK_DIR}/{ANCESTRY}_{GENE}.all_variants_toKeep.txt"

            plinkGLM = [
                f"{TOOLS_DIR}/plink2",
                "--pfile", inputPfile,
                "--keep", samplesToKeep,
                "--glm", "hide-covar",
                "cols=+a1count,+a1freqcc,+a1countcc,+beta,+totallele,+totallelecc,+gcountcc",
                "--extract", extract,
                "--pheno", covariate,
                "--pheno-name", "AAO",
                "--no-input-missing-phenotype",
                "--hwe", "0.0001",
                "--mac", "1",
                "--adjust",
                "--ci", "0.95",
                "--covar", covariate,
                "--covar-variance-standardize",
                "--covar-name", "SEX,PHENO,PC1,PC2,PC3,PC4,PC5",
                "--keep-if", "PHENO==2",
                "--out", outputGLM
            ]
            ## Pheno==2 will be constant and excluded from covariates, you will need this to only keep PD cases

            ## Run plink2 command as bash subprocess
            subprocess.run(plinkGLM, check=True)
    except:
        print(f"Error with {ANCESTRY} run. Check error logs")
        continue

In [ ]:
# only include ancestries that did not fail in the final results file
import pandas as pd

for ANCESTRY in ["AAC", "AFR", "AJ", "EAS", "EUR", "SAS"]:

    ancestry = ANCESTRY
    CHROM = 'chr21'
    GENE = 'ITSN1'
    PREFIX = f"{ANCESTRY}"
    
    OUTPUT_DIR = f'/home/121825_ITSN1_Results/PLINK_GLM_files'
    
    # join glm and adjusted glm file
    glm = pd.read_csv(f"{OUTPUT_DIR}/{PREFIX}.cases_only.AAO.glm.linear", sep="\t")
    glm_adjust = pd.read_csv(f"{OUTPUT_DIR}/{PREFIX}.cases_only.AAO.glm.linear.adjusted", sep="\t", usecols=["ID", "BONF"])
    
    full_glm_results = pd.merge(glm, glm_adjust, on="ID", how="left")
    full_glm_results.insert(0, "Ancestry", ancestry)
    full_glm_results.to_csv(f"{OUTPUT_DIR}/{ancestry}_{GENE}_AAO_glm_results.txt", sep="\t", index=False)

In [ ]:
# combine into one file
! cat {OUTPUT_DIR}/*AAO_glm_results.txt > {OUTPUT_DIR}/merged_ITSN1_AAO_glm.txt